# Chapter 2 · Completeness and repeated records

**Learning outcomes:** Detect and count missing information and exact repeated rows; select original evidence; explain the effect; and save clear check and issue records.

**Checks in this chapter:** CHK01 Name available; CHK02 Containment date available; CHK03 Exact repeated source rows.

## Block 1 · Establish the source and reporting tools

**Objective:** Load the source and understand the two small user-defined functions used to record checks.

**Expectation:** Conditions remain visible in the notebook. The functions count and save results without changing raw.

**Where it comes from:** The agreed reporting structure introduced in Chapter 1.

**Objects used here:** `raw` is the source DataFrame. `summarise_check` returns a dictionary; `save_check` writes the dictionary and evidence to CSV files.

**Method reference:** [read_csv()](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [dict](https://docs.python.org/3/library/stdtypes.html#dict) · [DataFrame()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) · [to_csv()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html)


In [2]:
import pandas as pd
from pathlib import Path
from inspection_helpers import summarise_check, save_check

# Start from the same unchanged teaching file in every notebook.
# Paths are relative to the study-guide folder opened in VS Code.
DATASET_PATH = Path("Data/wildfires.csv")
raw = pd.read_csv(DATASET_PATH, low_memory=False)

# An empty table cannot show that values passed a check.
if len(raw) == 0:
    raise ValueError("No records were loaded. Check Data/wildfires.csv before continuing.")

print("Source file:", DATASET_PATH)
print("Imported records:", len(raw))
print("Columns:", raw.columns.tolist())


FileNotFoundError: [Errno 2] No such file or directory: 'Data\\wildfires.csv'

**Two functions, with clear jobs:**

| Function | Inputs | What happens / what comes back |
| --- | --- | --- |
| `summarise_check(check_id, check_name, total, flagged, untested=0)` | Reference, title and record counts | Returns one dictionary containing Total, Checked, Passed, Flagged, Untested and the check result. |
| `save_check(check, evidence, finding, effect, next_action)` | That dictionary, an evidence DataFrame and three written explanations | Saves the check, issue, full evidence and a small evidence preview to `Outputs`. It returns no result. |

`untested=0` is a default argument. Omit it when every record can enter the check.
If some records lack a prerequisite, supply their count. The calculation is:
**Checked = Total − Untested; Passed = Checked − Flagged**.

The functions are defined once in `inspection_helpers.py` beside the notebooks. Their complete
code and explanation are included at the end of this notebook for study. You can follow the
teaching blocks without opening another resource. Later chapters import these same functions.


## Block 2 · Practise missing-value decisions

**Objective:** See what isna checks and why it is not a test for every kind of missing information.

**Expectation:** For availability, a value recognised by Pandas as missing is Flagged. Availability can inspect every record.

**Where it comes from:** Q3 needs a containment date; Q4 needs a name when identifying a record.

**Objects used here:** `missing_demo` is a new four-record example. It does not change or inspect raw.

**Method reference:** [DataFrame()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) · [isna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html) · [sum()](https://pandas.pydata.org/docs/reference/api/pandas.Series.sum.html) · [loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html)


In [1]:
# These invented names contrast a missing value with empty and space-filled text.
missing_demo = pd.DataFrame({"fire_name": ["Oak", None, "", "   "]})
demo_name_absent = missing_demo["fire_name"].isna()
missing_demo["Name is missing"] = demo_name_absent
display(missing_demo)
print("Recognised missing values:", demo_name_absent.sum())


NameError: name 'pd' is not defined

**Read the result:** Only None is recognised as missing in this demonstration. An empty string or spaces is text here and needs a separate text check. CSV import can recognise empty fields as missing.



**Pause and explain:** Why might a populated cell still contain no useful name?

<details>
<summary>Suggested answer — open after discussing</summary>

A string containing only spaces has characters but no useful name. Chapter 3 examines outside spaces and trimming.

</details>


## Block 3 · CHK01 · Find records without a name

**Objective:** Measure absent names and preserve the original affected records.

**Expectation:** A reported name is available for Q4. Example: Oak meets the availability rule; a missing value is Flagged.

**Where it comes from:** Our plan to identify and display named records. The source permits absent names, so this is an analytical limitation, not automatically a source error.

**Objects used here:** `raw` is the original source table. `name_absent` will contain one missing-name decision per source row. The count, evidence and log are created together below.

**Method reference:** [isna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html) · [sum()](https://pandas.pydata.org/docs/reference/api/pandas.Series.sum.html) · [loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) · [copy()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html) · [DataFrame()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)


In [ ]:
# isna asks one question per record: is its name absent?
name_absent = raw["fire_name"].isna()
absent_name_count = name_absent.sum()

# Availability can examine every imported row. Missing rows are Flagged, not
# Untested, because absence is exactly what this check is asking about.
check_name = summarise_check("CHK01", "Name available", len(raw), absent_name_count)
display(pd.DataFrame([check_name]))

# Select original values using the same condition used for the count.
# copy creates a separate evidence table so additional evidence columns cannot alter raw.
evidence_names = raw.loc[name_absent, ["fire_name", "state", "fire_year"]].copy()
evidence_names["Record outcome"] = "Flagged"
display(evidence_names.head(5))  # Only the display is shortened.

save_check(
    check_name, evidence_names,
    finding="Some reported records have no fire name.",
    effect="Q4 cannot show a name for those records; other available fields can still support other questions.",
    next_action="Investigate whether names can be recovered. Retain usable records for questions that do not need a name."
)


**Read the result:** The check table counts the full source. The evidence contains every missing-name record; head(5) displays five examples. A fully named file still produces a check row and an empty evidence table.

**Next action:** The proposed action is written guidance. No names have been filled and no source rows have been removed.

**Pause and explain:** Why should missing names not automatically cause rows to be deleted from the size-by-cause analysis?

<details>
<summary>Suggested answer — open after discussing</summary>

That question uses size and cause. A missing name alone does not make those values unavailable.

</details>


## Block 4 · CHK02 · Find records without a containment date

**Objective:** Find the availability limitation affecting duration analysis.

**Expectation:** A containment-date value is present. A timestamp meets this availability rule; a missing value is Flagged.

**Where it comes from:** Q3 needs both discovery and containment. Date readability and order are separate later checks.

**Objects used here:** `raw` is the unchanged source. `containment_absent` will supply the count, original evidence and CHK02 summary within this block.

**Method reference:** [isna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html) · [sum()](https://pandas.pydata.org/docs/reference/api/pandas.Series.sum.html) · [loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) · [copy()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html) · [DataFrame()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)


In [ ]:
containment_absent = raw["contained_date"].isna()
absent_containment_count = containment_absent.sum()
check_containment = summarise_check(
    "CHK02", "Containment date available", len(raw), absent_containment_count
)
display(pd.DataFrame([check_containment]))

# Keep both endpoints: the evidence should show why the duration is unavailable.
evidence_containment = raw.loc[
    containment_absent, ["fire_name", "discovery_date", "contained_date"]
].copy()
evidence_containment["Record outcome"] = "Flagged"
display(evidence_containment.head(5))

save_check(
    check_containment, evidence_containment,
    finding="Some records have no containment-date value.",
    effect="Q3 cannot calculate duration for these records even when a discovery date is available.",
    next_action="Seek the missing dates from a reliable source, or define and report a subset with both usable dates."
)


**Read the result:** An available value passes availability even if it later proves unreadable as a date. A missing containment date is Flagged here because this rule asks about its presence.

**Next action:** Do not invent dates to make a calculation possible. Investigate recovery or document a suitable subset.

**Pause and explain:** What happens to the same missing-date record when we check date order?

<details>
<summary>Suggested answer — open after discussing</summary>

Its date order is Untested: a required endpoint is unavailable. Chapter 4 will record that limitation.

</details>


## Block 5 · CHK03 · Understand exact repeated rows

**Objective:** Distinguish repeated values from confirmed duplicate physical fires.

**Expectation:** Review source rows whose complete eight-field values also occur in another row.

**Where it comes from:** Repeated records can change reported counts and totals. This extract lacks an incident identifier, so exact matching is a review screen.

**Objects used here:** `repeat_demo` is a new three-record table showing the effect of keep=False. It is independent of raw.

**Method reference:** [DataFrame()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) · [duplicated()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html) · [sum()](https://pandas.pydata.org/docs/reference/api/pandas.Series.sum.html) · [loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html)


In [ ]:
repeat_demo = pd.DataFrame({
    "fire_name": ["Oak", "Oak", "Pine"],
    "fire_size": [3.0, 3.0, 3.0]
})
# keep=False includes both partners in the matching pair. We need to review
# the whole pair, not silently accept the first occurrence as authoritative.
repeated_demo_rows = repeat_demo.duplicated(keep=False)
repeat_demo["Needs repeat review"] = repeated_demo_rows
display(repeat_demo)
print("Matching-group records:", repeated_demo_rows.sum())


**Read the result:** Two records belong to a matching group. This count differs from a count of later copies: there is one later copy in this demonstration. CHK03 consistently counts all matching-group records.



**Pause and explain:** Would two identical fire names alone establish an exact repeated source row?

<details>
<summary>Suggested answer — open after discussing</summary>

No. This check compares every source field. Even a complete match still needs investigation of record identity.

</details>


In [ ]:
# Compare all eight original source fields. The same name alone is insufficient.
repeated_source_rows = raw.duplicated(keep=False)
repeated_record_count = repeated_source_rows.sum()
check_repeats = summarise_check(
    "CHK03", "No exact repeated source rows", len(raw), repeated_record_count
)
display(pd.DataFrame([check_repeats]))

# Include every matching partner and all fields that established the match.
# Original row labels remain available in the saved source_row column.
evidence_repeats = raw.loc[repeated_source_rows].copy()
evidence_repeats["Record outcome"] = "Flagged"
display(evidence_repeats.head(5))

save_check(
    check_repeats, evidence_repeats,
    finding="Some source rows have the same complete values as other source rows.",
    effect="Repeated reported values may affect Q1 counts and Q2 totals if they represent redundant records.",
    next_action="Review the matching rows and original source identifiers before deciding whether any record is redundant."
)


**Read the result:** The printed count is the number of original records in matching groups. The CSV retains all partners; the five-row display may show only part of a group.



**Pause and explain:** Can we remove every flagged row and claim that we have kept one record per fire?

<details>
<summary>Suggested answer — open after discussing</summary>

No. That would remove all partners, and these fields do not establish physical-fire identity.

</details>


## Block 6 · Read one saved check beside its issue and evidence

**Objective:** See how one finding connects three different kinds of table.

**Expectation:** The check log records the check; the issue log explains any concern; evidence contains its source records.

**Where it comes from:** The reporting model taught in Chapter 1. CHK02 was saved in Block 4.

**Objects used here:** The three CSV files below are saved CHK02 results. They are loaded here, so no earlier in-memory variable is needed.

**Method reference:** [read_csv()](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [head()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html)


In [ ]:
# Loading results lets us study them without repeating the missing-date check.
saved_containment_check = pd.read_csv("Outputs/checks/CHK02.csv")
saved_containment_issue = pd.read_csv("Outputs/issues/CHK02.csv")
containment_evidence_preview = pd.read_csv("Outputs/previews/CHK02.csv")

display(saved_containment_check)
display(saved_containment_issue)
display(containment_evidence_preview)


**Read the result:** A check file has one row. Its issue file has one row when there is a concern and zero rows when none is found. The preview keeps up to five Flagged and five Untested examples; the evidence CSV is complete.



**Pause and explain:** Why are hundreds of thousands of evidence rows still consistent with one issue row?

<details>
<summary>Suggested answer — open after discussing</summary>

The issue describes one type of finding. Each evidence row is one record supporting it.

</details>


## Block 7 · Explain the consequences and check your counting

**Objective:** Explain findings in terms of the planned questions.

**Student activity:** Explain one possible effect and one justified next action for each:
missing name, missing containment date, and exact repeated rows.

**Suggested reasoning:** A missing name limits identifying a named record but need not limit size
comparison. A missing containment date prevents duration for that record. Exact repeated rows
may affect counts, but the source needs review before identifying redundant incidents.

**Counting check you can explain:** For complete evidence containing one row per concern,
evidence rows = Flagged + Untested for that check. Our three availability/repetition checks have
zero Untested records, so their evidence row count equals Flagged. Do not apply this relationship
to a five-row preview, a summary table, or evidence grouped by category.

**Saved outputs:** CHK01–CHK03 each have a check CSV, an issue CSV, a complete evidence CSV and a short preview CSV
under `Outputs`. The final reporting chapter reads them; it does not reimplement these checks.
The next chapter asks whether present values are written in a useful and interpretable form.


## Study reference · How the two reporting functions work

This is a reading reference, outside the main lecture blocks. The executable definitions live
in `inspection_helpers.py`; the notebooks import them. The code below is shown here so that
you can study it without changing files during a lesson.

`summarise_check` turns the supplied counts into one dictionary. Its if/elif/else decides the
check summary while leaving the Untested count visible. `save_check` creates folders, writes
that dictionary as one row, saves original evidence with its source labels, saves a balanced preview, and writes an issue
only when Flagged or Untested records need attention. A passed check still gets a check row.
Rerunning a check replaces its four result files, including an older issue if it has now passed.

These functions do not decide whether a source value is correct, assign severity, or perform
cleaning. Those decisions require the question, the rule and the original evidence.


```python
"""Two small reporting functions used by Chapters 2, 3 and 4.

The notebooks explain the arguments and show this code in their reference section.
Inspection conditions remain in the notebooks. These functions count and save
results; they never change the original dataset or choose a cleaning treatment.
"""

from pathlib import Path
import pandas as pd


def summarise_check(check_id, check_name, total, flagged, untested=0):
    """Return one dictionary describing one record-level check.

    total: all imported records in the check's scope.
    flagged: records checked that need review.
    untested: records that lacked information required for this particular check.
    """
    total = int(total)
    flagged = int(flagged)
    untested = int(untested)
    checked = total - untested
    passed = checked - flagged

    # A positive flagged count needs review. No checked records means Untested.
    # Otherwise Passed refers to checked records; untested records stay visible.
    if flagged > 0:
        result = "Flagged"
    elif checked == 0:
        result = "Untested"
    else:
        result = "Passed"

    return {
        "Check ID": check_id,
        "Check": check_name,
        "Total records": total,
        "Checked records": checked,
        "Passed records": passed,
        "Flagged records": flagged,
        "Untested records": untested,
        "Check result": result,
    }


def save_check(check, evidence, finding, effect, next_action):
    """Save one check, its issue entry (if needed), and its complete evidence.

    check: the dictionary returned by summarise_check.
    evidence: original records selected in the notebook, with one row per concern.
    finding, effect, next_action: explanations written by the student.
    """
    check_id = check["Check ID"]
    check_folder = Path("Outputs/checks")
    issue_folder = Path("Outputs/issues")
    evidence_folder = Path("Outputs/evidence")
    preview_folder = Path("Outputs/previews")
    for folder in [check_folder, issue_folder, evidence_folder, preview_folder]:
        folder.mkdir(parents=True, exist_ok=True)

    # [check] is a list containing one dictionary: it creates one check-log row.
    check_table = pd.DataFrame([check])
    check_table.to_csv(check_folder / f"{check_id}.csv", index=False)

    # Preserve the original Pandas row label as a normal CSV column. It locates
    # a row in this unchanged import; it is not a fire ID or a physical line number.
    evidence_path = evidence_folder / f"{check_id}.csv"
    # Write to a temporary filename first. Publish the completed file only after
    # the CSV writer has closed it, so another reader cannot see a partial export.
    pending_evidence = evidence_path.with_suffix(".tmp")
    evidence.to_csv(pending_evidence, index=True, index_label="source_row")
    pending_evidence.replace(evidence_path)

    # A balanced preview shows both kinds of concern when both occur. These are
    # examples, not counts: keep up to five Flagged and five Untested records.
    flagged_examples = evidence.loc[evidence["Record outcome"] == "Flagged"].head(5)
    untested_examples = evidence.loc[evidence["Record outcome"] == "Untested"].head(5)
    preview = pd.concat([flagged_examples, untested_examples])
    preview.to_csv(preview_folder / f"{check_id}.csv", index=True, index_label="source_row")

    issue_columns = [
        "Check ID", "Finding", "Flagged records", "Untested records",
        "Effect on analysis", "Next action", "Evidence CSV",
    ]
    # Missing prerequisites can limit analysis even when tested records passed.
    # A fully passed check needs no issue. Its saved issue file then has headings
    # and zero rows, so rerunning a check also replaces any older issue entry.
    if check["Flagged records"] > 0 or check["Untested records"] > 0:
        issue = {
            "Check ID": check_id,
            "Finding": finding,
            "Flagged records": check["Flagged records"],
            "Untested records": check["Untested records"],
            "Effect on analysis": effect,
            "Next action": next_action,
            "Evidence CSV": evidence_path.as_posix(),
        }
        issue_table = pd.DataFrame([issue], columns=issue_columns)
    else:
        issue_table = pd.DataFrame(columns=issue_columns)
    issue_table.to_csv(issue_folder / f"{check_id}.csv", index=False)

    # This summary keeps the scope and the saved evidence count visible in class.
    print(f"{check_id}: {check['Check result']} for {check['Checked records']:,} checked records")
    print(f"Flagged: {check['Flagged records']:,}; Untested: {check['Untested records']:,}")
    print(f"Complete evidence rows saved: {len(evidence):,} -> {evidence_path}")
```

**Method reference:** [dict](https://docs.python.org/3/library/stdtypes.html#dict) · [DataFrame()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) · [to_csv()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html) · [Path](https://docs.python.org/3/library/pathlib.html#pathlib.Path)
